# Quantum Phase Estimation
**Note**: This assignment has been adapted from the Xanadu Quantum Codebook [[1](#refs)].

The central idea for the QPE subroutine is to successively apply controlled-$U$ at powers of 2. 
Every such application of $U$ (e.g., $U^{2^0}$, $U^{2^1}$, $U^{2^2}$, $\cdots$), moves the decimal point in the *fractional binary* representation one place to the right, i.e.,

$$
U^{2^k} \vert \psi \rangle = \exp {\left[2 \pi i \theta_1\theta_2\cdots \theta_k.\theta_{k+1}\cdots \theta_t \right]} \vert \psi \rangle. 
$$

Any $e^{2\pi i \theta}$ that forms the *integral* part ($\theta_1$, $\theta_2$ in the above example) and not the *fractional* part ($\theta_3 \cdots \theta_t$) will be 1 (since $e^{2\pi i \theta} = 1$ for $\theta=0$ or 1), giving us the following state:

$$
U^{2^2} \vert \psi \rangle = \exp {\left[2 \pi i 0.\theta_{3}\cdots \theta_t \right]}\vert \psi \rangle . 
$$

In general, 

$$
U^{2^k} \vert \psi \rangle = \exp {\left[2 \pi i 0.\theta_{k+1}\cdots \theta_t \right]}\vert \psi \rangle . 
$$

Let us take a look at the full QPE circuit:

![phase kickback](../images/qpe_img.png "Full QPE circuit")

The circuit should have a familiar structure. Instead of the Hadamard-oracle-Hadamard circuit structure, we have the Hadamard-oracle-QFT ${^\dagger}$ (note that we can also interpret this as a QFT-oracle-QFT ${^\dagger}$  structure, but since the initial operation is applied to qubits in the $\vert 0 \rangle$ state, we can simply use the Hadamard transform in lieu of the QFT).

The state of the qubits at *Step 2* after applying the controlled unitaries to the estimation wires is

$$
\frac{1}{2^{t/2}} \left( \vert 0 \rangle + e^{2 \pi i 0.\theta_t} \vert 1 \rangle\right)
\left( \vert 0 \rangle + e^{2 \pi i 0.\theta_{t-1}\theta_t} \vert 1 \rangle\right) \cdots
\left( \vert 0 \rangle + e^{2 \pi i 0.\theta_1\theta_2\cdots\theta_t} \vert 1 \rangle \right).
$$


This state may be familiar: it is exactly the state obtained when a QFT is applied to a set of qubits,

$$ U_{QFT}\vert x_1 x_2 \cdots x_n \rangle = \frac{1}{\sqrt N}\left[ \left(\vert 0 \rangle + e^{2 \pi i 0.x_n} \vert 1 \rangle \right) \left(\vert 0 \rangle + e^{2 \pi i 0.x_{n-1}x_n} \vert 1 \rangle\rangle \right) \cdots \left( \vert 0 \rangle + e^{2 \pi i 0.x_1x_2\cdots x_n} \vert 1 \rangle \right) \right],  $$

where  $n = t$, $x = \theta$, and $N = 2^t$.

Therefore, an inverse QFT can be performed (*Step 3*) to get back

$$
\vert \theta_1 \theta_2 \cdots \theta_t \rangle.
$$

Measuring this state will give us precisely the phase we are looking for!

Implement the QPE subroutine to find the phase, and therefore, the eigenvalue of a unitary matrix, $U$ with eigenvector $\vert \psi \rangle$ in the following way:

1. Implement a helper function that computes $U^{2^k}$.
2. Write a subroutine to apply these $U^{2^k}$ controlled on a set of estimation wires.
3. Implement the circuit end-to-end to obtain the phase and eigenvalue.
4. Implement QPE for a special case.

In [1]:
import pennylane as qml
from pennylane import numpy as np

# global variable for wire count
num_wires = 10

dev = qml.device('default.qubit', wires=num_wires)

## Task 1
Given a unitary matrix $U$, compute the value of a higher power, $U^{2^k}$. You can use the ```matrix_power``` function from NumPy's linear algebra library.

In [2]:
def U_power_2k(unitary, k):
    """ 
    Computes U at a power of 2k (U^2k)
    Args: 
        unitary (array [complex]): A unitary matrix
    
    Returns: 
        array [complex]: U raised to the power of 2k
        
    """
    return np.linalg.matrix_power(unitary, 2**k)



## Task 2 
Implement a subroutine that applies the sequence of $U^{2^k}$ unitaries on the *target wires* controlled on the *estimation wires*. 

In [3]:
def apply_controlled_powers_of_U(unitary, estimation_wires, target_wires):
    """ 
    Args: 
        unitary (array [complex]): A unitary matrix
        estimation_wires (Sequence[int]): Estimation wires
        target_wires (Sequence[int] or int): Target wires
    Returns:
        None
    """
    if isinstance(estimation_wires, int):
        estimation_wires = [estimation_wires]
    else:
        estimation_wires = list(estimation_wires)

    if isinstance(target_wires, int):
        target_wires = [target_wires]
    else:
        target_wires = list(target_wires)

    # Match QFT wire ordering by assigning U^(2^0) to the last estimation wire.
    for k, control_wire in enumerate(reversed(estimation_wires)):
        U_k = U_power_2k(unitary, k)
        qml.ControlledQubitUnitary(U_k, wires=[control_wire, *target_wires])


## Task 3
Implement the QPE subroutine given a unitary, a set of estimation wires, and a set of target wires. Additionally, the function `prepare_eigenvector` which prepares an eigenvector of the unitary operator is also given to you below. To prepare other eigenvectors, modify this function. To implement the QFT$^\dagger$, you can make use of [PennyLane's template for QFT](https://pennylane.readthedocs.io/en/stable/code/api/pennylane.QFT.html) and [`qml.adjoint`](https://pennylane.readthedocs.io/en/stable/code/api/pennylane.adjoint.html)

In [4]:
def prepare_eigenvector(target_wires):
    """ 
    Args:
        target_wires (Sequence[int] or int): Target wires
    Returns:
        None
    """
    qml.PauliX(wires=target_wires)

@qml.qnode(dev)
def qpe(unitary, estimation_wires, target_wires):
    """ Estimate the phase for a given unitary.
    Args:
        unitary (array[complex]): A unitary matrix
        estimation_wires (Sequence[int]): Estimation wires
        target_wires (Sequence[int] or int): Target wires
    Returns:
        probs (array[float]): Probabilities on the estimation wires.
    """
    if isinstance(estimation_wires, int):
        estimation_wires = [estimation_wires]
    else:
        estimation_wires = list(estimation_wires)

    if isinstance(target_wires, int):
        target_wires = [target_wires]
    else:
        target_wires = list(target_wires)

    # Superposition
    for wire in estimation_wires:
        qml.Hadamard(wires=wire)
    
    # Prepare eigenvector
    prepare_eigenvector(target_wires)
    
    # Apply controlled powers of U
    apply_controlled_powers_of_U(unitary, estimation_wires, target_wires)
    
    # Apply the inverse QFT
    qml.adjoint(qml.QFT)(wires=estimation_wires)

    return qml.probs(wires=estimation_wires)


## Task 4

Check your work. For each case, determine the exact number of estimation and target wires required to get the correct phase angle with a 100% probability. The following are the unitaries:

$$
U_1 = T = \begin{bmatrix}
1 & 0 \\
0 & e^{\frac{i\pi}{4}} \\
\end{bmatrix}, \text{Eigenvector} = |1\rangle
$$

$$
U_2 =  \begin{bmatrix}
1 & 0 \\
0 & e^{\frac{7\pi i }{8}} \\
\end{bmatrix}, \text{Eigenvector} = |1\rangle
$$

$$
U_3 = \begin{bmatrix}
1 & 0 \\
0 & e^{\frac{i\pi}{8}} \\
\end{bmatrix}, \text{Eigenvector} = |1\rangle
$$

$$
U_4 = X = \begin{bmatrix}
0 & 1 \\
1 & 0 \\
\end{bmatrix}, \text{Any eigenvector}
$$

In [5]:
def qpe_wire_configuration(unitary, phase):
    """Return estimation and target wires for an exact QPE phase."""
    target_wire_count = int(np.log2(unitary.shape[0]))
    if 2 ** target_wire_count != unitary.shape[0]:
        raise ValueError("Unitary dimension must be a power of 2.")

    max_estimation_wires = num_wires - target_wire_count
    phase = phase % 1

    for estimation_wire_count in range(1, max_estimation_wires + 1):
        scaled_phase = phase * (2 ** estimation_wire_count)
        if np.isclose(scaled_phase, round(scaled_phase)):
            estimation_wires = list(range(estimation_wire_count))
            target_wires = list(
                range(estimation_wire_count, estimation_wire_count + target_wire_count)
            )
            return estimation_wires, target_wires

    raise ValueError("Phase does not terminate in binary on the current device.")


#### 4.1: $U_1$

In [6]:
U_1 = qml.T.compute_matrix()
estimation_wires, target_wires = qpe_wire_configuration(U_1, phase=1 / 8)
probs_1 = qpe(U_1, estimation_wires, target_wires)
print(probs_1)


[1.04495873e-32 1.00000000e+00 1.59422951e-32 3.56297040e-33
 3.89907714e-33 4.09260113e-33 3.72017131e-33 5.44075209e-33]


#### 4.2: $U_2$

In [7]:
U_2 = np.array([[1, 0], [0, np.exp((7*np.pi*1j/8))]])
estimation_wires, target_wires = qpe_wire_configuration(U_2, phase=7 / 16)
probs_2 = qpe(U_2, estimation_wires, target_wires)
print(probs_2)


[2.55185718e-33 3.49648192e-33 3.44925137e-33 8.86301344e-33
 1.29518789e-32 2.17280419e-32 9.11410761e-32 1.00000000e+00
 7.40038581e-32 2.42775238e-32 1.01523743e-32 1.01319079e-32
 5.44075209e-33 5.15155884e-33 2.69286173e-33 4.92043695e-33]


#### 4.3: $U_3$

In [8]:
U_3 = np.array([[1, 0], [0, np.exp((np.pi*1j/8))]])
estimation_wires, target_wires = qpe_wire_configuration(U_3, phase=1 / 16)
probs_3 = qpe(U_3, estimation_wires, target_wires)
print(probs_3)


[1.76342961e-32 1.00000000e+00 6.46108033e-33 3.71657792e-33
 4.39352769e-33 1.60269365e-33 8.03166970e-34 1.17466623e-33
 5.89816045e-34 3.37989289e-33 3.95257068e-34 4.86932292e-34
 5.41667797e-34 5.40536679e-34 1.35502747e-33 5.75293768e-33]


#### 4.4: $U_4$

In [9]:
def prepare_eigenvector(target_wires):
    """ Prepare the |-⟩ eigenvector of the X gate.
    Args:
        target_wires (Sequence[int] or int): Target wires
    Returns:
        None
    """
    qml.PauliX(wires=target_wires)
    qml.Hadamard(wires=target_wires)


In [10]:
U_4 = qml.PauliX.compute_matrix()
estimation_wires, target_wires = qpe_wire_configuration(U_4, phase=1 / 2)
probs_4 = qpe(U_4, estimation_wires, target_wires)
print(probs_4)


[0. 1.]


## Task 5
Given the probabilities on the estimation wires, estimate the phase associated with a unitary, when the eigenvector is prepared in the state $\vert 1 \rangle$.

In [11]:
def estimate_phase(probs):
    """ 
    Args: 
        probs (array[float]): Probabilities on the estimation wires.
    
    Returns:
        float: the estimated phase   
    """
    max_index = int(np.argmax(probs))
    return max_index / len(probs)


## Task 6
Estimate the phase for $U_1$, $U_2$, $U_3$, $U_4$ using results (probabilities) of QPE from above.

In [12]:
print(estimate_phase(probs_1))


0.125


In [13]:
print(estimate_phase(probs_2))


0.4375


In [14]:
print(estimate_phase(probs_3))


0.0625


In [15]:
print(estimate_phase(probs_4))


0.5


## Task 7
Use [PennyLane's template for QPE](https://pennylane.readthedocs.io/en/stable/code/api/pennylane.QuantumPhaseEstimation.html) to calculate the phase of the $T$ gate using Quantum Phase Estimation. Additionally, you may use this to verify that your work was indeed correct by running it for U_2, U_3 and U_4. 

In [16]:
def prepare_one_state(target_wires):
    qml.PauliX(wires=target_wires)

def prepare_x_minus_state(target_wires):
    qml.PauliX(wires=target_wires)
    qml.Hadamard(wires=target_wires)

def qpe_template_probs(unitary, estimation_wires, target_wires, prepare_eigenvector_fn):
    if isinstance(estimation_wires, int):
        estimation_wires = [estimation_wires]
    else:
        estimation_wires = list(estimation_wires)

    if isinstance(target_wires, int):
        target_wires = [target_wires]
    else:
        target_wires = list(target_wires)

    dev = qml.device("default.qubit", wires=max(estimation_wires + target_wires) + 1)

    @qml.qnode(dev)
    def circuit():
        prepare_eigenvector_fn(target_wires)
        qml.QuantumPhaseEstimation(
            unitary,
            target_wires=target_wires,
            estimation_wires=estimation_wires,
        )
        return qml.probs(wires=estimation_wires)

    return circuit()

expected_phases = {
    "U_1": estimate_phase(probs_1),
    "U_2": estimate_phase(probs_2),
    "U_3": estimate_phase(probs_3),
    "U_4": estimate_phase(probs_4),
}

template_cases = [
    ("U_1", qml.T.compute_matrix(), [0, 1, 2], [3], prepare_one_state),
    ("U_2", np.array([[1, 0], [0, np.exp((7 * np.pi * 1j / 8))]]), [0, 1, 2, 3], [4], prepare_one_state),
    ("U_3", np.array([[1, 0], [0, np.exp((np.pi * 1j / 8))]]), [0, 1, 2, 3], [4], prepare_one_state),
    ("U_4", qml.PauliX.compute_matrix(), [0], [1], prepare_x_minus_state),
]

for name, unitary, estimation_wires, target_wires, prepare_eigenvector_fn in template_cases:
    probs = qpe_template_probs(unitary, estimation_wires, target_wires, prepare_eigenvector_fn)
    phase = estimate_phase(probs)
    print(f"{name}: template={phase}, earlier={expected_phases[name]}")


U_1: template=0.125, earlier=0.125
U_2: template=0.4375, earlier=0.4375
U_3: template=0.0625, earlier=0.0625
U_4: template=0.5, earlier=0.5


## <a name="refs"></a>References

[1] C. Albornoz, G. Alonso, M. Andrenkov, P. Angara, A. Asadi, A. Ballon, S. Bapat, L. Botelho, I. De Vlugt, O. Di Matteo, P. Downing, P. Finlay, A. Fumagalli, A. Gardhouse, N. Girard, A. Hayes, J. Izaac, R. Janik, T. Kalajdzievski, N. Killoran, I. Kurečić, O. Landon-Cardinal, D. Nino, A. Otto, C. Pere, J. Pickering, J. Soni, D. Wakeham. (2023) Xanadu Quantum Codebook.